### Decoder Only Transformer from scratch 

In [24]:
import tensorflow as tf
tf.__version__

'2.20.0'

In [25]:
# Attention Mechanism
def scaled_dot_product_attention(q, k, v, mask=None):
   
    #QK^T
    matmul_qk = tf.linalg.matmul(q, k, transpose_b=True)

    #Scaling
    dk = tf.cast(tf.shape(k)[-1], tf.float32)
    scaled_logits = matmul_qk / tf.math.sqrt(dk)

    #Masking
    if mask is not None:
        mask = tf.cast(mask, tf.float32)
        scaled_logits += (mask * -1e9)

    # 4.Softmax Normalization
    attention_weights = tf.nn.softmax(scaled_logits, axis=-1)

    # 5. Multiply with V
    output = tf.linalg.matmul(attention_weights, v)

    return output, attention_weights

In [26]:
#Multihead Attention

class MultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads):
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.depth = d_model // num_heads

        # Linear projections
        self.wq = tf.keras.layers.Dense(d_model)
        self.wk = tf.keras.layers.Dense(d_model)
        self.wv = tf.keras.layers.Dense(d_model)

        # Final output projection
        self.dense = tf.keras.layers.Dense(d_model)

    def split_heads(self, x, batch_size):
        """
        Input shape:  (batch, seq_len, d_model)
        Output shape: (batch, num_heads, seq_len, depth)
        """
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, x, mask=None):
        batch_size = tf.shape(x)[0]

        # 1. Linear projections
        q = self.wq(x)  # (B, S, d_model)
        k = self.wk(x)
        v = self.wv(x)

        # 2. Split into heads
        q = self.split_heads(q, batch_size)  # (B, H, S, depth)
        k = self.split_heads(k, batch_size)
        v = self.split_heads(v, batch_size)

        # 3. Scaled dot-product attention
        scaled_attention,attention_weights =scaled_dot_product_attention(q,k,v,mask)

        # 4. Transpose back
        scaled_attention = tf.transpose(scaled_attention, perm=[0, 2, 1, 3])
        # (B, S, H, depth)

        # 5. Concatenate heads
        concat_attention=tf.reshape(scaled_attention,(batch_size,-1,self.d_model))
        # (B, S, d_model)

        # 6. Final linear layer
        output = self.dense(concat_attention)

        return output, attention_weights

In [27]:
#Testing
# checking for shapes of input and output
batch_size = 2
seq_len = 5
d_model = 128
num_heads = 4

x = tf.random.uniform((batch_size, seq_len, d_model))

mha = MultiHeadAttention(d_model, num_heads)

output, weights = mha(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("Attention weights shape:", weights.shape)

Input shape: (2, 5, 128)
Output shape: (2, 5, 128)
Attention weights shape: (2, 4, 5, 5)


In [28]:
# positional Encoding

def positional_encoding(seq_len, d_model):

    position = tf.range(seq_len, dtype=tf.float32)[:, tf.newaxis] # Position indices (S, 1)

    i = tf.range(d_model, dtype=tf.float32)[tf.newaxis, :]  # Dimension indices (1, d_model)

    angle_rates = 1 / tf.pow(10000.0,(2 * (tf.floor(i / 2))) / tf.cast(d_model, tf.float32))

    angle_rads = position * angle_rates  # (S, d_model)

    sines = tf.sin(angle_rads[:, 0::2])
    cosines = tf.cos(angle_rads[:, 1::2])

    # Interleave sin and cos properly
    pos_encoding = tf.reshape(tf.stack([sines, cosines], axis=-1),(seq_len, d_model))

    # Add batch dimension
    pos_encoding = pos_encoding[tf.newaxis, ...]

    return pos_encoding

class PositionalEncoding(tf.keras.layers.Layer):
    def __init__(self, max_seq_len, d_model):
        super().__init__()
        self.pos_encoding = positional_encoding(max_seq_len, d_model)

    def call(self, x):
        seq_len = tf.shape(x)[1]

        # Add positional encoding to embeddings
        return x + tf.cast(self.pos_encoding[:, :seq_len, :], x.dtype)

In [29]:
#testing
batch_size = 2
seq_len = 5
d_model = 128

x = tf.random.uniform((batch_size, seq_len, d_model))

pos_layer = PositionalEncoding(max_seq_len=100, d_model=d_model)
output = pos_layer(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)

Input shape: (2, 5, 128)
Output shape: (2, 5, 128)


In [30]:
#Feed Forward Neural Network
class FeedForwardNetwork(tf.keras.layers.Layer):
    def __init__(self, d_model, dff):
        super().__init__()

        self.dense1 = tf.keras.layers.Dense(dff, activation='relu')
        self.dense2 = tf.keras.layers.Dense(d_model)

    def call(self, x):
        x = self.dense1(x)   # (B, S, hidden layer neuron)
        x = self.dense2(x)   # (B, S, d_model)
        return x
    
#Look-Ahead Mask (Causal Mask)
def create_look_ahead_mask(seq_len):
    #Shape: (S, S)
    mask = 1 - tf.linalg.band_part(tf.ones((seq_len, seq_len), dtype=tf.float32), -1, 0)
    return mask


In [31]:

#Decoder Block
class DecoderBlock(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, dff, dropout_rate=0.1):
        super().__init__()

        # Multi-head self-attention
        self.mha = MultiHeadAttention(d_model, num_heads)

        # Feed Forward Network
        self.ffn = FeedForwardNetwork(d_model, dff)

        # Layer Normalization
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)

        # Dropout
        self.dropout1 = tf.keras.layers.Dropout(dropout_rate)
        self.dropout2 = tf.keras.layers.Dropout(dropout_rate)

    def call(self, x, training=False, mask=None):
        
        #Masked Multi-Head Attention
        attn_output, attn_weights = self.mha(x, mask)
        attn_output = self.dropout1(attn_output, training=training)

        # Add + LayerNorm
        out1 = self.layernorm1(x + attn_output)

       
        #Feed Forward Network
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)

        # Add + LayerNorm
        out2 = self.layernorm2(out1 + ffn_output)

        return out2, attn_weights

In [ ]:
# final complete architecture
class GPT(tf.keras.Model):
    def __init__(self, vocab_size, max_seq_len,
                 num_layers, d_model, num_heads, dff, dropout_rate=0.1):
        super().__init__()

        self.d_model = d_model

        # Token Embedding
        self.embedding = tf.keras.layers.Embedding(vocab_size, d_model)

        # Positional Encoding
        self.pos_encoding = PositionalEncoding(max_seq_len, d_model)

        # Decoder Blocks
        self.decoder_blocks = []
        for _ in range(num_layers):
            self.decoder_blocks.append(
                DecoderBlock(d_model, num_heads, dff, dropout_rate)
            )

        # Dropout
        self.dropout = tf.keras.layers.Dropout(dropout_rate)

        # Final Output Layer (logits)
        self.final_layer = tf.keras.layers.Dense(vocab_size)

    def call(self, x, training=False):
        """
        Args:
            x: (B, S) → token IDs
        Returns:
            logits: (B, S, vocab_size)
        """

        seq_len = tf.shape(x)[1]

       
        #Create Look-Ahead Mask (FIXED SHAPE)
       
        mask = create_look_ahead_mask(seq_len)           # (S, S)
        mask = mask[tf.newaxis, tf.newaxis, :, :]        # (1, 1, S, S)

        #Embedding
        
        x = self.embedding(x)                            # (B, S, d_model)

        # Scale embeddings (important)
        x *= tf.math.sqrt(tf.cast(self.d_model, x.dtype))

        # 3. Positional Encoding
        x = self.pos_encoding(x)

        x = self.dropout(x, training=training)

        # 4. Decoder Blocks
        attention_weights = {}

        for i, block in enumerate(self.decoder_blocks):
            x, attn = block(x, training=training, mask=mask)
            attention_weights[f"block_{i+1}"] = attn

        # 5. Final Linear Layer
        logits = self.final_layer(x)                     # (B, S, vocab_size)

        return logits, attention_weights